# Generating Elaborations with pretrained LLaMA instruction-tuned model

This notebook focuses on generating elaborations using the **pretrained LLaMA instruction-tuned model**, utilizing three versions of the **few-shot prompt**: the short version (with three examples), the medium version (with six examples) and the long version (with nine examples). 

Elaborations are generated employing **beam search with 4 beams**.

Users can follow the elaboration generation and evaluation process step by step within the notebook. Alternatively, if you prefer to execute the full pipeline automatically, you can run the following scripts via terminal:

**For elaboration generation:**

```bash
python generate_elaborations.py --model llama-instruct
``` 
This assumes that the required model and dataset are correctly placed in their respective directories.

# Load data

In [8]:
from dataset_utils import load_dataset_from_csv

ds_type = "c2s"
setting = "target-phrase"
num_examples = "n3"
output_name = f"{ds_type}-{setting}"

dataset = load_dataset_from_csv(ds_type, setting)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['doc_num', 'source_text', 'label_text', 'elaboration_sentence', 'contextual_specificity_rating', 'target_sentence_target'],
        num_rows: 1046
    })
    validation: Dataset({
        features: ['doc_num', 'source_text', 'label_text', 'elaboration_sentence', 'contextual_specificity_rating', 'target_sentence_target'],
        num_rows: 132
    })
    test: Dataset({
        features: ['doc_num', 'source_text', 'label_text', 'elaboration_sentence', 'contextual_specificity_rating', 'target_sentence_target'],
        num_rows: 116
    })
})


# Load the model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch.cuda.empty_cache()
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.2-3B-Instruct', cache_dir="../models/llama/") 
model =  AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-3B-Instruct', cache_dir="../models/llama/", device_map ={'':torch.cuda.current_device()})

# Prompt design

#### Examples

In [56]:
from prompt_utils import examples_dict
print(examples_dict.keys())

dict_keys(['Definition', 'Example', 'Analogy', 'Background', 'Reason', 'Contrast', 'Result', 'Speculation', 'Supplementation'])


In [9]:
from prompt_utils import formatting_prompt_func, base_prompt_fewshot

formatted_test_dataset = formatting_prompt_func(examples=dataset["test"], EOS="", base_prompt=base_prompt_fewshot, setting=setting, num_examples=num_examples)
print(formatted_test_dataset[0])

### User: You are an expert in clarifying unclear,complex term or concept in a given text. Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in plain English for a given context text. The tone should be plain and simple! Do not add any comments to your answer! 
For example:

context text: 'She teaches at the University of Utah. In 1974, Wiessner recorded conversations among the Ju/'hoansi Bushmen. They live in a vast area of 124 miles in southwestern Africa. Their lives have changed since the 1970s.'
target_phrase='Bushmen'
Assistant: 'The Bushmen are a group of people who hunt animals and gather wild berries and plants to eat.'

context text: 'There are differences in how the increases would work. The differences have to do with how the cost of living would be measured. The minimum wage in Alaska would be based on prices in Alaska. South Dakota would raise the minimum wage based on changes to a national measure of the cost of

### BASE

#### Short version

In [3]:
from prompt_utils import examples_dict, base_prompt, insert_examples, create_user_message
# set the pad_token for llama 3.2 3B
tokenizer.pad_token = tokenizer.eos_token
EOS_TOKEN = tokenizer.eos_token

versions = {
    "n3":['Definition','Example','Background'],
    "n6":['Definition','Example','Background', 'Supplementation', 'Analogy', 'Speculation'],
    "n9":['Definition', 'Example', 'Analogy', 'Background', 'Reason', 'Contrast', 'Result', 'Speculation', 'Supplementation']
}

filtered_dict = {key: value for key, value in examples_dict.items() if key in versions[num_examples]}

def formatting_test_prompts_func(examples):
    
    contexts = examples["source_text"]
    texts = []
    for context in contexts:
        text = base_prompt.format(insert_examples(filtered_dict, setting),create_user_message(context, setting)) 
        texts.append(text)
    return texts

formatted_test_dataset = formatting_test_prompts_func(dataset["test"])

In [4]:
print(formatted_test_dataset[0])

### User: You are an expert in clarifying unclear or complex terms and concepts in a given text. Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in plain English for a given context text. The tone should be plain and simple! Do not add any comments to your answer! 
For example:

context text: 'She teaches at the University of Utah. In 1974, Wiessner recorded conversations among the Ju/'hoansi Bushmen. They live in a vast area of 124 miles in southwestern Africa. Their lives have changed since the 1970s.'
target_phrase='Bushmen'
Assistant: 'The Bushmen are a group of people who hunt animals and gather wild berries and plants to eat.'

context text: 'There are differences in how the increases would work. The differences have to do with how the cost of living would be measured. The minimum wage in Alaska would be based on prices in Alaska. South Dakota would raise the minimum wage based on changes to a national measure of the c

#### Long version

In [17]:
from prompt_utils import examples_dict, base_prompt, insert_examples, examples_dict_from_validation_dataset

# set the pad_token for llama 3.2 3B
tokenizer.pad_token = tokenizer.eos_token
EOS_TOKEN = tokenizer.eos_token

def create_user_message(context):
    return f"Return an explanation sentence for the following context text: '{context}'."


def formatting_test_prompts_func(examples):
    contexts = examples["source_text"]
    texts = []
    for context in contexts:
        text = base_prompt.format(insert_examples(examples_dict_from_validation_dataset),create_user_message(context))
        texts.append(text)
    return texts

formatted_test_dataset = formatting_test_prompts_func(dataset["test"])
print(formatted_test_dataset[1])

### User: You are an expert in clarifying unclear or complex terms and concepts in a given text. Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in plain English for a given context text. The tone should be plain and simple! Do not add any comments to your answer! 
For example:

context text: 'Together they slowed down the large group of trucks carrying the machine parts. They were not able to stop the trucks for long, though. This one was carrying a giant water evaporator. After being blocked, it continued on its way when police arrested 20 of the protesters.'
Assistant: 'Trucks that travel in groups are known as a convoy.'

context text: 'There are differences in how the increases would work. The differences have to do with how the cost of living would be measured. The minimum wage in Alaska would be based on prices in Alaska. South Dakota would raise the minimum wage based on changes to a national measure of the cost of l

### Masked version

In [138]:
# set the pad_token for llama 3.2 3B
tokenizer.pad_token = tokenizer.eos_token
EOS_TOKEN = tokenizer.eos_token

def create_user_message(context):
    return f"Return the explanation sentence that could replace the `<explanatory sentence>` tag in the following text: '{context}'."

test_alpaca_prompt = """### User: Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in a plain English that could replace the <explanatory sentence> tag in a given context text. The tone should be plain and simple! Do not add any comments to your answer! 
For example: 
context text: 'The environment is essential for sustaining life, providing clean air, water, and fertile soil. <explanatory sentence> Protecting the environment ensures a healthier planet for future generations. '
assistant: 'Fertile soil is ideal for growing plants.'

context text: 'Japan is known for its rich cultural heritage and advanced technology. Its landscapes range from serene cherry blossom gardens to towering Mount Fuji. <explanatory sentence>'
assistant: 'Mount Fuji is the tallest mountain in Japan.'

{}\n### Assistant:"""

def formatting_test_prompts_func(examples):
    contexts = examples["source_text"]
    texts = []
    for context in contexts:
        text = test_alpaca_prompt.format(create_user_message(context)) 
        texts.append(text)
    return texts

formatted_test_dataset = formatting_test_prompts_func(dataset["test"])

### Specifying subject

In [83]:
# set the pad_token for llama 3.2 3B
tokenizer.pad_token = tokenizer.eos_token
EOS_TOKEN = tokenizer.eos_token

def create_user_message_subject(context, subject):
    return f"Return the explanation sentence for the following context text: '{context}'. The explanation sentence should refer to the {subject}."
print(tokenizer.eos_token )

test_alpaca_prompt = """### User: Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in a plain English for a given context text. The tone should be plain and simple! Do not add any comments to your answer! 
For example: 
context text: 'The environment is essential for sustaining life, providing clean air, water, and fertile soil. Protecting it ensures a healthier planet for future generations. '
subject='protecting the environment'
assistant: 'This includes reducing pollution and conserving resources.'

context text: 'Japan is known for its rich cultural heritage and advanced technology. Its landscapes range from serene cherry blossom gardens to towering Mount Fuji.'
subject='Mount Fuji'
assistant: 'Mount Fuji is the tallest mountain in Japan.'

{}\n### Assistant:"""

def formatting_test_prompts_func(examples):
    contexts = examples["source_text"]
    subjects = examples["subject"]
    texts = []
    for context, subject in zip(contexts, subjects):
        text = test_alpaca_prompt.format(create_user_message_subject(context, subject)) 
        texts.append(text)
    return texts

formatted_test_dataset = formatting_test_prompts_func(dataset["test"])

<|eot_id|>


### Specifying the target phrase

In [5]:
from prompt_utils import examples_dict, base_prompt, insert_examples, create_user_message

# set the pad_token for llama 3.2 3B
tokenizer.pad_token = tokenizer.eos_token
EOS_TOKEN = tokenizer.eos_token

def create_user_message_target(context, target):
    return f"Return the explanation sentence for the following context text: '{context}'. The explanation sentence should specifically clarify the {target}."

versions = {
    "n3":['Definition','Example','Background'],
    "n6":['Definition','Example','Background', 'Supplementation', 'Analogy', 'Speculation'],
    "n9":['Definition', 'Example', 'Analogy', 'Background', 'Reason', 'Contrast', 'Result', 'Speculation', 'Supplementation']
}

filtered_dict = {key: value for key, value in examples_dict.items() if key in versions[num_examples]}

def formatting_test_prompts_func(examples):
    contexts = examples["source_text"]
    targets = examples["target_sentence_target"]
    texts = []
    for context, target in zip(contexts, targets):
        text = base_prompt.format(insert_examples(filtered_dict, setting), create_user_message(context, setting, target)) 
        texts.append(text)
    return texts

formatted_test_dataset = formatting_test_prompts_func(dataset["test"])

In [6]:
print(formatted_test_dataset[0])

### User: You are an expert in clarifying unclear or complex terms and concepts in a given text. Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in plain English for a given context text. The tone should be plain and simple! Do not add any comments to your answer! 
For example:

context text: 'She teaches at the University of Utah. In 1974, Wiessner recorded conversations among the Ju/'hoansi Bushmen. They live in a vast area of 124 miles in southwestern Africa. Their lives have changed since the 1970s.'
target_phrase='Bushmen'
Assistant: 'The Bushmen are a group of people who hunt animals and gather wild berries and plants to eat.'

context text: 'There are differences in how the increases would work. The differences have to do with how the cost of living would be measured. The minimum wage in Alaska would be based on prices in Alaska. South Dakota would raise the minimum wage based on changes to a national measure of the c

### Specifying the target sentence for clarification

In [37]:
# set the pad_token for llama 3.2 3B
tokenizer.pad_token = tokenizer.eos_token
EOS_TOKEN = tokenizer.eos_token

def create_user_message_target(context, target):
    return f"Return the explanation sentence for the following context text: '{context}'. The explanation sentence should specifically clarify the target_sentence={target}"

test_alpaca_prompt = """### User: You are an expert in clarifying unclear or complex terms and concepts in a given text. Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in plain English for a given context text. The tone should be plain and simple! Do not add any comments to your answer! 
For example: 
context text: 'Japan is known for its rich cultural heritage and advanced technology. Its landscapes range from cherry blossom gardens to towering Mount Fuji.'
target_sentence='Japan is known for its rich cultural heritage and advanced technology.'
Assistant: This heritage includes traditional arts like tea ceremony or calligraphy. 

context text: 'One of the most thrilling events in winter sports is ski jumping. Ski jumping is a winter sport where athletes glide down a ramp and jump to achieve maximum distance.'
target_sentence='Ski jumping is a winter sport where athletes glide down a ramp and jump to achieve maximum distance.'
Assistant: As they glide down, they gain speed, which helps them jump higher into the air.

{}\n### Assistant:"""

def formatting_test_prompts_func(examples):
    contexts = examples["source_text"]
    targets = examples["target_sentence_4o"]
    texts = []
    for context, target in zip(contexts, targets):
        text = test_alpaca_prompt.format(create_user_message_target(context, target)) 
        texts.append(text)
    return texts

formatted_test_dataset = formatting_test_prompts_func(dataset["test"])

### Specifying both the target phrase/ subject and the target sentence

In [48]:
# set the pad_token for llama 3.2 3B
tokenizer.pad_token = tokenizer.eos_token
EOS_TOKEN = tokenizer.eos_token

def create_user_message_target(context, target,target_sentence):
    return f"Return the explanation sentence for the following context text: '{context}'. The explanation sentence should specifically clarify the target_sentence='{target_sentence}' by referring to the {target}."

test_alpaca_prompt = """### User: You are an expert in clarifying unclear or complex terms and concepts in a given text. Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in plain English for a given context text. The tone should be plain and simple! Do not add any comments to your answer! 
For example: 
context text: 'Japan is known for its rich cultural heritage and advanced technology. Its landscapes range from cherry blossom gardens to towering Mount Fuji.'
target_sentence='Japan is known for its rich cultural heritage and advanced technology.'
target_phrase='cultural heritage'
Assistant: This heritage includes traditional arts like tea ceremony or calligraphy. 

context text: 'One of the most thrilling events in winter sports is ski jumping. Ski jumping is a winter sport where athletes glide down a ramp and jump to achieve maximum distance.'
target_sentence='Ski jumping is a winter sport where athletes glide down a ramp and jump to achieve maximum distance.'
target_phrase='glide down'
Assistant: As they glide down, they gain speed, which helps them jump higher into the air.

{}\n### Assistant:"""

def formatting_test_prompts_func(examples):
    contexts = examples["source_text"]
    targets = examples["target_sentence_target"] # subject
    target_sents = examples["target_sentence_4o"]
    texts = []
    for context, target, target_sent in zip(contexts, targets, target_sents):
        text = test_alpaca_prompt.format(create_user_message_target(context, target, target_sent)) 
        texts.append(text)
    return texts

formatted_test_dataset = formatting_test_prompts_func(dataset["test"])

### Specifying both the target phrase (from the target sentence) and the target sentence

In [5]:
# set the pad_token for llama 3.2 3B
tokenizer.pad_token = tokenizer.eos_token
EOS_TOKEN = tokenizer.eos_token

alpaca_prompt = """### User: Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in a plain English for a given context text. The tone should be plain and simple! {}\n### Assistant: {}"""

def create_user_message_target(context, target, target_sentence):
    return f"Return the explanation sentence for the following context text: '{context}'. The explanation sentence should specifically clarify the {target_sentence} by referring to the {target}."
print(tokenizer.eos_token )

def formatting_prompts_func(examples):
    contexts = examples["source_text"]
    targets = examples["target_sentence_target"]
    target_sents = examples["target_sentence_4o"]
    elab_sentences = examples["elaboration_sentence"]
    texts = []
    for context, target, target_sent, elab_sent in zip(contexts, targets, target_sents, elab_sentences):
        text = alpaca_prompt.format(create_user_message_target(context, target, target_sent), elab_sent) + EOS_TOKEN
        texts.append(text)
    return texts

test_alpaca_prompt = """### User: Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in a plain English for a given context text. The tone should be plain and simple! {}\n### Assistant:"""

def formatting_test_prompts_func(examples):
    contexts = examples["source_text"]
    targets = examples["target_sentence_target"]
    target_sents = examples["target_sentence_4o"]
    texts = []
    for context, target, target_sent in zip(contexts, targets, target_sents):
        text = test_alpaca_prompt.format(create_user_message_target(context, target, target_sent)) 
        texts.append(text)
    return texts

formatted_test_dataset = formatting_test_prompts_func(dataset["test"])

<|end_of_text|>


# Generate predictions

https://github.com/NielsRogge/Transformers-Tutorials/blob/master/Mistral/Supervised_fine_tuning_(SFT)_of_an_LLM_using_Hugging_Face_tooling.ipynb

In [8]:
from transformers import pipeline, StoppingCriteria, StoppingCriteriaList
import torch

class RefinedEndSentenceStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, sentence_end_tokens):
        super().__init__()
        self.tokenizer = tokenizer
        self.sentence_end_token_ids = [
            self.tokenizer.convert_tokens_to_ids(token) for token in sentence_end_tokens
        ]
        self.eos_token_id = tokenizer.eos_token_id 

    def is_valid_stop(self, input_ids):
        if len(input_ids[0]) < 2:
            return False 
        last_token_id = input_ids[0, -1].item()
        second_last_token_id = input_ids[0, -2].item()

        last_token = self.tokenizer.decode([last_token_id])
        second_last_token = self.tokenizer.decode([second_last_token_id])

        
        if (
            last_token in [".", "!", "?"]  
            and len(second_last_token) > 1  
            and not second_last_token.isupper()  # check if it's not "U.S." or similar
        ):
            return True
        return last_token_id == self.eos_token_id

    def __call__(self, input_ids, scores, **kwargs):
        return self.is_valid_stop(input_ids)


sentence_end_tokens = [".","\n","!", "?"]
stopping_criteria = StoppingCriteriaList([RefinedEndSentenceStoppingCriteria(tokenizer, sentence_end_tokens)])

## Example

In [8]:
import torch
import random 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

example = random.choice(formatted_test_dataset)

inputs = tokenizer(
    example, 
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=512  
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=32,  
        min_length=10,
        do_sample=False,  
        temperature=None, 
        top_p=None, 
        num_return_sequences=1,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        stopping_criteria=stopping_criteria
    )

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated_text)
response = extract_response(generated_text)
print("Extracted Response:", response)

### User: Your task is to generate exactly ONE short concise explanation sentence (made up of around 10 words or fewer) in a plain English for a given context text. 
The tone should be plain and simple! Return the explanation sentence for the following context text: 'Brown was a black teenager without a weapon who was shot by a white police officer. He was killed in August in Ferguson, Missouri, near St. Louis. The shooting set off nearly nightly protests and violence. The black community felt that Brown wouldn't have been killed if he was white.'.
### Assistant: The officer was not charged with a crime.
Extracted Response: The officer was not charged with a crime.


## Generate predictions

In [7]:
from tqdm.notebook import tqdm
import pandas as pd
from dataset_utils import create_results_df

def extract_response(text, prefix = "### Assistant:"):
    if prefix in text:
        return text.split(prefix, 1)[1].strip()
    return None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
model.config.use_cache = True

sentence_end_tokens = [".","\n","!", "?"]
stopping_criteria = StoppingCriteriaList([RefinedEndSentenceStoppingCriteria(tokenizer, sentence_end_tokens)])

def generate_predictions(dataset, formatted_test_dataset, ds_type, setting, num_examples):

    output_name = f"{ds_type}-{setting}"
    search_type = {"beam-search":{"num_beams":4, "early_stopping":True, 
                              "filename":f"../data/gen_predictions/predictions_llama-instruct-few-shot-{output_name}-{num_examples}.csv"},
              "greedy":{"num_beams":1, "early_stopping":False,
                        "filename":f"../data/gen_predictions/predictions_llama-instruct-few-shot-{output_name}-greedy-{num_examples}.csv"}
    }

    for search_t in search_type.keys():
    
        df_results = create_results_df(dataset)
    
        for idx, row in tqdm(df_results.iterrows(),total=len(df_results)):
            if row["pred_elaboration"]=="":
                inputs = tokenizer(
                    formatted_test_dataset[idx], 
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=2500 
                ).to(device)
                
                with torch.no_grad():
                    output_ids = model.generate(
                        input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"],
                        max_new_tokens=32,  
                        min_length=10,
                        do_sample=False, 
                        temperature=None,  
                        top_p=None,
                        num_beams = search_type[search_t]["num_beams"],
                        early_stopping = search_type[search_t]["early_stopping"],
                        num_return_sequences=1,
                        no_repeat_ngram_size=3,
                        eos_token_id=tokenizer.eos_token_id,
                        pad_token_id=tokenizer.eos_token_id,
                        stopping_criteria=stopping_criteria
                    )
                
                generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
                response = extract_response(generated_text) 
                df_results.at[idx,"pred_elaboration"] = response
        
        df_results.to_csv(search_type[search_t]["filename"], index=False)
        print(f"Saved {search_type[search_t]['filename']}")